# GLIDE — Quickstart

**GLIDE** turns a small set of true evaluation labels (e.g. human annotations) plus and a large pool of proxy evaluation labels (e.g. LLM-as-judge annotations) into a **bias-corrected metric with a valid confidence interval**.

| Ingredient | Role |
|-----------|------|
| True labels | Ground truth — accurate but slow and expensive |
| Proxy labels | Proxy — cheap and fast but biased |

## Installation

Before running the code below (see bottom for complete notebook), install **glide-py**:

```bash
pip install glide-py
```

In [ ]:
from glide.core.simulated import generate_dataset_binary
from glide.estimators.ppi import PPIMeanEstimator

## Step 1 — Assemble Your Data

A `Dataset` is a list of records, each record being a Python dict. Records come in two kinds:

- **Labeled** — have both `y_true` and `y_proxy`
- **Unlabeled** — have only `y_proxy`

Any key names work, pass them to `estimate()` at call time.

To run this notebook end-to-end, generate a synthetic dataset with the built-in helper:

We simulate 100 groundtruth plus 900 proxy-labeled conversations. 

The true rate is 15% and the proxy rate is 10% (biased low).

In [ ]:
dataset = generate_dataset_binary(
    n=100,            # labeled records
    N=900,            # unlabeled records
    true_mean=0.15,   # true rate
    proxy_mean=0.10,  # proxy rate (biased)
    correlation=0.70,
    random_seed=0,
)

print(f"Dataset : {len(dataset)} records")
print()
print(f"Labeled sample   -> {dataset[0]}")
print(f"Unlabeled sample -> {dataset[-1]}")

## Step 2 — Estimate the Metric

`PPIMeanEstimator.estimate()` corrects the proxy's bias using the labeled subset, then applies the correction across the full dataset.

When calling `estimate()`, we specify which columns of the dataset correspond to the true and proxy labels using the `y_true_field` and `y_proxy_field` parameters.

In [ ]:
result = PPIMeanEstimator().estimate(
    dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    metric_name="Evaluated metric", # e.g. an LLM's hallucination rate
    confidence_level=0.95,
)

print(result.summary())

The effective sample size is the number of true labels needed to achieve the same confidence with true labels only.

## Step 3 — Read the Result

`InferenceResult` contains the point estimate, standard error and confidence interval.

In [ ]:
# Point estimate and confidence interval
print(f"Estimate : {result.result.mean:.1%}")
print(f"Std err  : {result.result.std:.4f}")
print(f"95% CI   : [{result.result.lower_bound:.1%}, \
      {result.result.upper_bound:.1%}]")
print()

`InferenceResult` also allows you to perform hypothesis testing on the estimated distribution. For example, we can test the 95% confidence interval.

In [ ]:
# Test H0: hallucination rate = 10% (the judge's claim)
z, p, _ = result.result.test_null_hypothesis(
    h0_value=0.10,
    alternative="larger",  # H1: true rate > 10%
)
print("H0: rate = 10%  |  H1: rate > 10%")
print(f"z = {z:.2f}  |  p = {p:.4f}")
print("Reject H0 — the rate is significantly above 10%." 
      if p < 0.05 else "Cannot reject H0 at the 5% level.")

## Using Your Own Data Format

Replace `generate_dataset_binary` with any list of dicts. Labeled records must have both `y_true` and `y_proxy`; unlabeled records need only `y_proxy`.

```python
import json
import pandas as pd
import polars as pl

# From a JSON file
with open("annotations.json") as f:
    records = json.load(f)
dataset = Dataset(records)

# From a pandas DataFrame
df = pd.read_csv('your_data.csv')
records = df.to_dict(orient="records")
dataset = Dataset(records)

# From a polars DataFrame
df = pl.read_csv('your_data.csv')
records = df.to_dicts()
dataset = Dataset(records)

# From a numpy array
array = np.zeros((100,5))
records = [{f'field_{j}':array[i,j] 
            for j in range(array.shape[1])} 
            for i in range(array.shape[0])]
dataset = Dataset(records)


```

Then you can define an estimator (e.g. PPI) and pass your true and proxy column names in the snippet below.

```
ppi_estimator = PPIMeanEstimator()
result = ppi_estimator.estimate(
    dataset,
    y_true_field=...,
    y_proxy_field=...,
    metric_name="Evaluated metric", # e.g. an LLM's output Hallucination rate
    confidence_level=0.95,
)

print(result.summary())
```
